### Autoloader

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS ansh_catalog.bronze.autovolume

In [0]:
df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", "/Volumes/ansh_catalog/bronze/autovolume/destination/checkpoint/") \
  .option("cloudFiles.schemaEvolutionMode", "rescue") \
  .load("/Volumes/ansh_catalog/bronze/autovolume/raw/")

In [0]:
df.writeStream.format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", "/Volumes/ansh_catalog/bronze/autovolume/destination/checkpoint/") \
  .trigger(availableNow=True) \
  .start("/Volumes/ansh_catalog/bronze/autovolume/destination/data")

In [0]:
df = spark.read.format("delta")\
    .load("/Volumes/ansh_catalog/bronze/autovolume/destination/data")
display(df)

In [0]:
rescued_schema = StructType([StructField("discount", StringType(), True),
                             StructField("payment_method", StringType(), True)])

df = df.withColumn("rescued_struct", from_json(col("_rescued_data"), rescued_schema))
df = df.withColumn("rescued_discount", col("rescued_struct.discount"))
df = df.withColumn("rescued_payment_method", col("rescued_struct.payment_method"))
display(df)


### AddNewColumns and MergeSchema

In [0]:
df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", "/Volumes/ansh_catalog/bronze/autovolume/destination/checkpoint/") \
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
  .load("/Volumes/ansh_catalog/bronze/autovolume/raw/")

In [0]:
df.writeStream.format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", "/Volumes/ansh_catalog/bronze/autovolume/destination/checkpoint/") \
  .trigger(availableNow=True) \
  .option("mergeSchema", True) \
  .start("/Volumes/ansh_catalog/bronze/autovolume/destination/data/")

In [0]:
df = spark.read.format("delta")\
    .load("/Volumes/ansh_catalog/bronze/autovolume/destination/data")
display(df)